# ============================================
# 06_tep_braatz_downloader.ipynb
# Tennessee Eastman Process (TEP) Benchmark Dataset
# ============================================

## Dataset Title
Tennessee Eastman Process (TEP) Benchmark Dataset

## Source URL
Official repository page:  
https://github.com/camaramm/tennessee-eastman-profBraatz

Stable direct ZIP download URL:  
https://codeload.github.com/camaramm/tennessee-eastman-profBraatz/zip/refs/heads/master

## Purpose of This Dataset
This dataset is used as the real industrial process-control benchmark dataset in the PhD workflow.

It is especially useful for:
- process-control benchmarking
- anomaly and fault studies
- reinforcement learning environment design
- robust optimization experiments
- multivariate industrial state modeling

## Why We Need This Dataset
We need this dataset because it provides a classic industrial benchmark with both simulation code and structured training/testing data.

According to the public repository:
- it includes open-loop and closed-loop Tennessee Eastman simulation codes
- it includes training and testing data files
- it includes normal and fault operating conditions
- it is widely used for data-driven industrial process evaluation

This makes it highly suitable for:
- Notebook 6: Baseline industrial process dataset integration
- Notebook 7: NEO-Robust
- Notebook 8: NEO-Graph
- Notebook 9: NEO-RL
- Notebook 11: Full System Simulation

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset folders
3. Download the dataset ZIP from the stable GitHub codeload URL
4. Extract the ZIP
5. Identify usable files such as `.dat`, `.txt`, `.f`, and README files
6. Save a dataset manifest
7. Create a clean ZIP bundle
8. Save everything in Google Drive
9. Create a final package ZIP
10. Download that final package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 7: NEO-Robust
- Notebook 8: NEO-Graph
- Notebook 9: NEO-RL
- Notebook 11: Full System Simulation

## Notes
The public repository states that it contains the Tennessee Eastman open-loop and closed-loop simulation codes and the training/testing data files used for evaluating data-driven methods. Examples listed include `temain.f`, `temain_mod.f`, `teprob.f`, `d00.dat`, and `d00_te.dat`.

In [1]:
# ============================================
# 06_tep_braatz_downloader.ipynb
# FULL FIXED SINGLE CELL
# ============================================

import os
import json
import shutil
import zipfile
from pathlib import Path
import requests
from google.colab import drive, files

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "Tennessee Eastman Process (TEP) Benchmark Dataset"
SOURCE_PAGE = "https://github.com/camaramm/tennessee-eastman-profBraatz"
DOWNLOAD_URL = "https://codeload.github.com/camaramm/tennessee-eastman-profBraatz/zip/refs/heads/master"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("Download URL :", DOWNLOAD_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "06_tep_braatz"
raw_dir = dataset_root / "raw"
extracted_dir = dataset_root / "extracted"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, extracted_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", extracted_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Robust download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=600, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download dataset ZIP
# -----------------------------
download_path = raw_dir / "tennessee_eastman_profBraatz_master.zip"

if not download_path.exists():
    print("\nDownloading dataset ZIP...")
    download_file(DOWNLOAD_URL, download_path)
    print("Download complete:", download_path)
else:
    print("\nUsing cached file:", download_path)

if not download_path.exists():
    raise FileNotFoundError(f"Dataset ZIP not found: {download_path}")

print("ZIP size:", round(download_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 6. Extract ZIP
# -----------------------------
print("\nExtracting ZIP...")
with zipfile.ZipFile(download_path, "r") as zf:
    zf.extractall(extracted_dir)

print("Extraction complete:", extracted_dir)

# -----------------------------
# 7. Collect usable files
# -----------------------------
all_files = sorted([p for p in extracted_dir.rglob("*") if p.is_file()])

usable_files = []
for p in all_files:
    rel = str(p.relative_to(extracted_dir))
    if "__MACOSX" in rel or p.name.startswith("._"):
        continue
    if p.suffix.lower() in [".dat", ".txt", ".csv", ".json", ".f", ".for", ".md"]:
        usable_files.append(p)

print("\nTotal extracted files:", len(all_files))
print("Usable files:", len(usable_files))
for fp in usable_files[:40]:
    print(" -", fp)

# -----------------------------
# 8. Save manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "download_url": DOWNLOAD_URL,
    "downloaded_file": str(download_path),
    "extracted_dir": str(extracted_dir),
    "all_files": [str(p) for p in all_files],
    "usable_files": [str(p) for p in usable_files],
    "used_in_next_notebooks": [
        "Notebook 7: NEO-Robust",
        "Notebook 8: NEO-Graph",
        "Notebook 9: NEO-RL",
        "Notebook 11: Full System Simulation"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 9. Create clean bundle ZIP
# -----------------------------
clean_bundle_zip_path = bundle_dir / "tep_braatz_CLEAN_bundle.zip"

with zipfile.ZipFile(clean_bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in usable_files:
        if not fp.exists():
            continue
        arcname = str(fp.relative_to(extracted_dir))
        zf.write(fp, arcname=arcname)

print("Created clean bundle ZIP:", clean_bundle_zip_path)
print("Clean bundle size:", round(clean_bundle_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 10. Create full package ZIP in Google Drive
# -----------------------------
full_zip_path = dataset_root / "tep_braatz_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(download_path, arcname="raw/" + download_path.name)

    for file in extracted_dir.rglob("*"):
        if file.is_file():
            rel = str(file.relative_to(extracted_dir))
            if "__MACOSX" in rel or file.name.startswith("._"):
                continue
            zf.write(file, arcname="extracted/" + rel)

    zf.write(clean_bundle_zip_path, arcname="bundle/" + clean_bundle_zip_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP in Google Drive:", full_zip_path)
print("Full package size:", round(full_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 11. Copy final ZIP to /content for reliable PC download
# -----------------------------
runtime_copy = Path("/content/tep_braatz_FULL_PACKAGE.zip")
shutil.copy2(full_zip_path, runtime_copy)

print("Runtime backup created:", runtime_copy)
print("Runtime backup size:", round(runtime_copy.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 12. Final summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root         :", dataset_root)
print("Raw ZIP              :", download_path)
print("Extracted folder     :", extracted_dir)
print("Manifest             :", manifest_path)
print("Clean bundle ZIP     :", clean_bundle_zip_path)
print("Full package ZIP     :", full_zip_path)
print("Runtime backup ZIP   :", runtime_copy)
print("===================")

# -----------------------------
# 13. Download final package to PC
# -----------------------------
print("\nStarting browser download...")
files.download(str(runtime_copy))

Dataset Name : Tennessee Eastman Process (TEP) Benchmark Dataset
Source Page  : https://github.com/camaramm/tennessee-eastman-profBraatz
Download URL : https://codeload.github.com/camaramm/tennessee-eastman-profBraatz/zip/refs/heads/master
Mounted at /content/drive

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/06_tep_braatz
 - /content/drive/MyDrive/PhD_Thesis_Datasets/06_tep_braatz/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/06_tep_braatz/extracted
 - /content/drive/MyDrive/PhD_Thesis_Datasets/06_tep_braatz/bundle

Download complete: /content/drive/MyDrive/PhD_Thesis_Datasets/06_tep_braatz/raw/tennessee_eastman_profBraatz_master.zip
ZIP size: 4.4 MB

Extracting ZIP...
Extraction complete: /content/drive/MyDrive/PhD_Thesis_Datasets/06_tep_braatz/extracted

Total extracted files: 50
Usable files: 49
 - /content/drive/MyDrive/PhD_Thesis_Datasets/06_tep_braatz/extracted/tennessee-eastman-profBraatz-master/README.md
 - /

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>